<a href="https://colab.research.google.com/github/vhgauto/2026-Escuela-de-Primavera/blob/main/EdP2026_clorofila.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clorofila-a

Ejecutar en la terminal el siguiente comando:

`sudo apt install gdal-bin libabsl-dev libnode-dev libproj-dev`

In [ ]:
# 20min
install.packages("ranger", dependencies = TRUE)
install.packages("finetune", dependencies = TRUE)
install.packages("tidyverse", dependencies = TRUE)
install.packages("tidymodels", dependencies = TRUE)
install.packages("tidyterra", dependencies = TRUE)

# Preprocesamiento

Los paquetes principales son [terra](https://rspatial.org/) para el procesamiento de datos espaciales (rásters y vectores), [tidyterra](https://dieghernan.github.io/tidyterra/) facilita la confección de mapas, [tidymodels](https://tidymodels.tidymodels.org/) brinda una metodología de trabajo para la modelización y [tidyverse](https://tidyverse.tidyverse.org/) contiene una colección de paquetes para ciencia de datos.

In [ ]:
library(terra)
library(tidyterra)
library(tidymodels)
library(tidyverse)
tidymodels_prefer()

Se leen los archivos ráster de los recortes del Embalse San Roque.

In [ ]:
fechas <- c(20230522, 20250220, 20250604, 20251119)
l <- paste0(
  "https://github.com/vhgauto/2026-Escuela-de-Primavera/raw/refs/heads/main/tp_clorofila/raster_salida/",
  fechas,
  ".tif"
)
r_lista <- map(l, rast)

Leo el vector de puntos de muestreo.

In [ ]:
p <- vect("https://github.com/vhgauto/2026-Escuela-de-Primavera/raw/refs/heads/main/tp_clorofila/vectores/puntos.geojson")

Visualizo una de las fechas para verificar la correcta lectura de los datos, con los sitios de muestreo y sus nombres.

In [ ]:
r2 <- r_lista[[2]]
r2_fecha <- r2 |>
  sources() |>
  basename() |>
  ymd()

ggplot() +
  geom_spatraster_rgb(
    data = stretch(r2), r = 4, g = 3, b = 2, max_col_value = 150
  ) +
  geom_spatvector(data = p, shape = 21, fill = "red", color = "gold") +
  geom_spatvector_label(
    data = p,
    aes(label = sitio),
    size = 2.3,
    vjust = -.25,
    border.color = NA,
    fill = "white"
  ) +
  labs(title = r2_fecha) +
  coord_sf(expand = FALSE) +
  theme_void()

Como puede observarse, el sitio GAR se encuentra en una región estrecha del San Roque, por lo que se lo remueve y no será considerado en la etapa de modelización.

In [ ]:
p <- p[p$sitio != "GAR"]

A partir de los sitios y los ráster, se extraen las reflectancias de superficie de todos los recortes disponibles. Para facilitar la extracción de los valores de píxel, se crea la función f_reflect y se la aplica a cada elemento de la lista de recortes.

In [ ]:
f_reflect <- function(i) {
  terra::extract(i, p, bind = TRUE) |>
    as_tibble() |>
    mutate(fecha = ymd(basename(sources(i))), .before = 1)
}

reflect_tbl <- map(r_lista, f_reflect) |>
  list_rbind() |>
  select(fecha, sitio, starts_with("B"))

head(reflect_tbl)

Los datos de clorofila-a provienen de un archivo de texto (.csv) que contiene la fecha de muestreo, el nombre del sitio y la medición en μg/l.

In [ ]:
cla_tbl <- read_csv(
  file = "https://github.com/vhgauto/2026-Escuela-de-Primavera/raw/refs/heads/main/tp_clorofila/datos/clorofila.csv",
  show_col_types = FALSE
)

head(cla_tbl)

Finalmente, se combinan los datos espectrales con las mediciones de cla, a partir del nombre del sitio y la fecha. Para la modelización se seleccionan únicamente las bandas y el valor de `cla`.

In [ ]:
reflect_cla <- inner_join(reflect_tbl, cla_tbl, by = join_by(fecha, sitio)) |>
  select(cla, starts_with("B"))

head(reflect_cla)

# Correlaciones

Se calcula el índice espectral NDVI ya que está relacionado con la presencia de algas y ha sido utilizado en estudios previos para estimar clorofila-a.

$$
NDVI=\frac{B5-B4}{B5+B4}
$$

Creo una nueva base de datos que contiene el NDVI. Además, se calcula el cociente de bandas B5/B4, ya que se ha empleado en estudios similares.

In [ ]:
d <- reflect_cla |>
  mutate(
    ndvi = (B5 - B4) / (B5 + B4),
    B5_B4 = B5 / B4
  )

El coeficiente de correlación lineal (r) se obtiene entre `cla` y todas las bandas espectrales, el ndvi y el cociente `B5/B4`. Esto permite identificar aquellas variables que se asocien en mayor grado a la clorofila-a.

In [ ]:
pivot_longer(
  d,
  cols = -cla,
  values_to = "reflect",
  names_to = "banda"
) |>
  nest(.by = banda) |>
  mutate(r = map_dbl(data, ~ cor(.x$reflect, .x$cla))) |>
  mutate(pvalor = map(data, ~ cor.test(.x$reflect, .x$cla))) |>
  mutate(pvalor = map_dbl(pvalor, ~ tidy(.x)$p.value)) |>
  select(banda, r, pvalor) |>
  arrange(desc(r)) |>
  mutate(es_signif = pvalor < .05)

Esta tabla permite generar modelos candidatos con mejor criterio.

# División de Datos

Inicialmente, se dividen los datos en dos grupos: entrenamiento (75%) y validación (25%), respetando la distribución de la variable de salida (`cla`).

Dado que esta división es aleatoria, es relevante mantener la reproducibilidad a fin de que los resultados no varíen con cada prueba.

In [ ]:
set.seed(2000)
datos_split <- initial_split(d, strata = cla, prop = 3 / 4)
datos_train <- training(datos_split)
datos_test <- testing(datos_split)
datos_split

# Re-muestreo

Aplico bootstraping a los datos de entrenamiento, generando 25 nuevos conjuntos de datos que incluyen repeticiones.

In [ ]:
set.seed(2001)
datos_bootstrap <- bootstraps(datos_train, strata = cla, times = 25)

# Construcción de Modelos

El desarrollo del modelo requiere definir los especificaciones de los algoritmos y las variables de interés y explicativas.

## Mecanismos de los modelos

Se especifica la manera en la que se relacionan las variables explicativas con el parámetro de interés, indicando el mecanismo de asociación.

El modelo lineal utilizado para la regresión empleando la función `stats::lm`.

In [ ]:
lin_spec <- linear_reg(mode = "regression") |>
  set_engine("lm")

Modelo Random Forest (RF) requiere de parámetros constantes (hiperparámetros, HP) que surgen por iteración y no forman parte de los datos. Los valores que adoptan los HP afectan el desempeño de los modelos, requiriendo su optimización para encontrar el valor más adecuado.

Para el algoritmo RF, se optimiza la cantidad de árboles de decisión (`trees`). El modelo RF es creado por `ranger::ranger()` para lograr una regresión.

In [ ]:
rf_spec <- rand_forest(trees = tune()) |>
  set_engine("ranger") |>
  set_mode("regression")

## Recetas

Las recetas combinan la `cla`, nuestra variable de interés, con las variables explicativas permitiendo agregar etapas de modificación posteriores.

Para los modelos lineales se consideran las siguientes recetas: índice espectral ndvi y la banda B5 y el cociente `B5/B4`.

In [ ]:
receta_ndvi <- recipe(cla ~ ndvi, data = d)
receta_b5 <- recipe(cla ~ B5, data = d)
receta_b5_b4 <- recipe(cla ~ B5_B4, data = d)

Para los modelos de RF se definen dos recetas: todas las bandas espectrales disponibles (de `B1` a `B7`), y una selección de las mismas, conservando aquellas más relevantes para el fenómeno de estudio (`B3` y `B5`).

In [ ]:
receta_bandas <- recipe(cla ~ B1 + B2 + B3 + B4 + B5 + B6 + B7, data = d)
receta_select <- recipe(cla ~ B5 + B3, data = d)

## Workflow

Unifico todas las recetas y especificaciones de los modelos, indicando las combinaciones deseadas, en un `workflow_set`. Inicialmente se encuentra vacío previo a la etapa de entrenamiento.

In [ ]:
wf <- workflow_set(
  preproc = list(
    ndvi = receta_ndvi,
    b5 = receta_b5,
    b5_b4 = receta_b5,
    bandas = receta_bandas,
    select = receta_select
  ),
  models = list(
    lin = lin_spec,
    lin = lin_spec,
    lin = lin_spec,
    rf = rf_spec,
    rf = rf_spec
  ),
  cross = FALSE
)

# Entrenamiento

Se define la configuración empleada en la optimización de `trees`.

In [ ]:
race_ctrl <- finetune::control_race(
  save_pred = TRUE,
  parallel_over = "everything",
  save_workflow = TRUE
)

El entrenamiento del `workflow_set` requiere indicar los remuestreos, la configuración de la optimización del HP y el mecanismo a emplear.

Para reducir el tiempo de procesamiento dedicado a la optimización de `trees` se aplica la técnica de parada anticipada, que descarta potenciales valores de número de árboles si el desempeño empeora.

La etapa de entrenamiento es computacionalmente exigente, demorando varios minutos en procesar. El tiempo de ejecución dependerá de la cantidad de observaciones del conjunto de entrenamiento, del número de remuestreos, del mecanismo de optimización de HP, la cantidad de modelos a comparar y las características computacionales del sistema.

El objeto resultante `res` contiene cada modelo entrenado, y dada la configuración de entrenamiento (`race_ctrl`), cuenta con todos los remuestreos.

In [ ]:
# 4min
res <- wf |>
  workflow_map(
    fn = "tune_race_anova",
    seed = 2002,
    resamples = datos_bootstrap,
    grid = 50,
    control = race_ctrl
  )

# Selección del modelo

Exploro y comparo las métricas de desempeño de todos los modelos propuestos entrenados.

In [ ]:
f_entrenamiento <- function(metrica) {
  autoplot(
    res,
    rank_metric = metrica,
    metric = metrica,
    select_best = TRUE
  ) +
    geom_text(
      aes(y = mean, label = wflow_id),
      angle = 90,
      vjust = -.3,
      size = 5
    ) +
    coord_cartesian(clip = "off") +
    theme_bw() +
    theme(aspect.ratio = 1, legend.position = "none") +
    theme_sub_panel(background = element_blank()) +
    theme_sub_plot(background = element_blank())
}

In [ ]:
f_entrenamiento("rmse")

In [ ]:
f_entrenamiento("rsq")

El modelo `ndvi_lin` presentó el RMSE más bajo y el R$^2$ más alto. Se lo elije para la etapa de validación.

In [ ]:
res |>
  rank_results() |>
  pivot_wider(
    names_from = .metric,
    values_from = mean,
    id_cols = c(wflow_id, .config, rank, model)
  ) |>
  slice_min(order_by = rmse, n = 1, with_ties = FALSE) |>
  select(wflow_id, rank, rmse, rsq)

Se elije el mejor modelo, utilizando el RMSE como parámetro de selección.

In [ ]:
mejor_res <- res |>
  extract_workflow_set_result("ndvi_lin") |>
  select_best(metric = "rmse")

# Validación

Con el mejor modelo seleccionado, finalizo el `workflow` y procedo a la etapa de validación con un último ajuste empleando el conjunto de validación.

In [ ]:
val_res <- res |>
  extract_workflow("ndvi_lin") |>
  finalize_workflow(mejor_res) |>
  last_fit(split = datos_split)

Internamente, `val_res` contiene las métricas de desempeño de la validación, el modelo propiamente dicho, las estimaciones del conjunto de validación, etc.

Comparo las estimaciones de `cla` del conjunto de validación.

In [ ]:
collect_predictions(val_res) |>
  ggplot(aes(x = cla, y = .pred)) +
  geom_abline(color = "gray50", lty = 2) +
  geom_point(size = 4, alpha = .5, color = "dodgerblue") +
  annotate(geom = "text", x = I(.9), y = I(.94), label = "y=x", angle = 45) +
  coord_obs_pred(expand = TRUE) +
  labs(x = "cla observado (μg/L)", y = "cla estimado (μg/L)") +
  theme_bw() +
  theme_sub_panel(
    grid.major = element_line(linewidth = .1),
    background = element_blank()
  ) +
  theme_sub_plot(background = element_blank())

Las métricas finales del modelo resultan:

In [ ]:
collect_metrics(val_res) |>
  pivot_wider(names_from = .metric, values_from = .estimate) |>
  select(rmse, rsq)

Verifico que intervalo de confianza (95%) de la pendiente (`cla`) no incluya el cero entre las mediciones y las estimaciones.

In [ ]:
collect_predictions(val_res) |>
  lm(.pred ~ cla, data = _) |>
  confint(level = .95)

La expresión del modelo seleccionado, para `cla` en μg/L, resulta:

In [ ]:
const <- extract_fit_engine(val_res) |>
  tidy() |>
  pull(estimate) |>
  round(2)

cat("cla =", const[1], "+", const[2], "* NDVI")

# Mapa

Se aplica el modelo desarrollado a un producto satelital para inspeccionar la distribución espacial de clorofila-a sobre el Embalse San Roque.

Dado que el algoritmo solo puede aplicarse a píxeles de agua, se calcula el índice espectral MNDWI, para Landsat-8/9 definido como:

$$
MNDWI=\frac{B3-B6}{B3+B6}
$$

Se obtienen los píxeles de agua y se calcula el NDVI.

In [ ]:
mndwi <- (r2$B3 - r2$B6) / (r2$B3 + r2$B6)

m <- thresh(mndwi, method = "otsu")
m[isTRUE(m)] <- 1
m[isFALSE(m)] <- NA

agua <- r2 * m
agua_ndvi <- (agua$B5 - agua$B4) / (agua$B5 + agua$B4)
terra::set.names(agua_ndvi, "ndvi")

Se verifica que los píxeles de agua sean correctos con una visualización.



In [ ]:
plot(m, axes = FALSE, col = "dodgerblue", legend = FALSE, box = TRUE)
north(xy = "topleft", type = 1)

Específicamente, se aplica el modelo lineal sobre el ráster de agua, obteniendo un nuevo ráster con las estimaciones de `cla`. Se remueven valores extremos para favorecer la visualización.



In [ ]:
cla_pred <- terra::predict(agua_ndvi, extract_fit_engine(val_res))
cla_pred <- clamp(cla_pred, 0, 80)

Se crea un mapa para identificar la distribución espacial de clorofila-a.



In [ ]:
ggplot() +
  geom_spatraster(data = cla_pred) +
  scale_fill_whitebox_c(palette = "muted", breaks = scales::breaks_width(10)) +
  labs(title = r2_fecha, fill = "Clorofila-a\n(μg/L)") +
  theme_minimal() +
  theme_sub_legend(key.height = unit(45, "pt")) +
  theme_sub_panel(
    background = element_rect(fill = "grey95"),
    grid.major = element_line(linetype = 2, color = "grey80", linewidth = .3)
  ) +
  theme_sub_plot(background = element_blank()) +
  theme_sub_axis_left(text = element_text(angle = 90, hjust = .5))